In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline
import numpy as np

In [3]:
sequences = np.load("CMdata.npz")["seqs"]

min_len = 75
max_len = 95
filtered_sequences = [s for s in sequences if min_len <= len(s) <= max_len]
sequences = filtered_sequences
len(sequences)

8727

In [4]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(sequences))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'A', 2: 'C', 3: 'D', 4: 'E', 5: 'F', 6: 'G', 7: 'H', 8: 'I', 9: 'K', 10: 'L', 11: 'M', 12: 'N', 13: 'P', 14: 'Q', 15: 'R', 16: 'S', 17: 'T', 18: 'V', 19: 'W', 20: 'Y', 0: '.'}
21


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [11]:
# build the dataset
block_size = 70 # context length: how many characters do we take to predict the next one?

def build_dataset(sequences):
  X, Y = [], []

  for w in sequences:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(sequences)
#n1 = int(0.8*len(sequences))
n2 = int(0.9*len(sequences))

Xtr,  Ytr  = build_dataset(sequences[:n2])    # 90%
Xdev, Ydev = build_dataset(sequences[n2:])   # 10%


Xtr = Xtr.to(device)
Ytr = Ytr.to(device)
Xdev = Xdev.to(device)
Ydev = Ydev.to(device)


torch.Size([634200, 70]) torch.Size([634200])
torch.Size([70531, 70]) torch.Size([70531])


In [12]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 100 # the number of neurons in the hidden layer of the MLP
g = torch.Generator(device=device).manual_seed(1) # for reproducibility

C = torch.randn((vocab_size, n_embd), generator=g, device=device)

layers = [
  nn.Linear(n_embd * block_size, n_hidden, bias=False), nn.BatchNorm1d(n_hidden), nn.Tanh(),
  nn.Linear(           n_hidden, n_hidden, bias=False), nn.BatchNorm1d(n_hidden), nn.Tanh(),
  #nn.Linear(           n_hidden, n_hidden, bias=False), nn.BatchNorm1d(n_hidden), nn.Tanh(),
  #nn.Linear(           n_hidden, n_hidden, bias=False), nn.BatchNorm1d(n_hidden), nn.Tanh(),
  #nn.Linear(           n_hidden, n_hidden, bias=False), nn.BatchNorm1d(n_hidden), nn.Tanh(),
  nn.Linear(           n_hidden, vocab_size, bias=False), nn.BatchNorm1d(vocab_size),
]

for layer in layers:
  layer.to(device)

with torch.no_grad():
  # last layer: make less confident
  layers[-1].weight *= 0.1
  #layers[-1].weight *= 0.1
  # all other layers: apply gain
  for layer in layers[:-1]:
    if isinstance(layer, nn.Linear):
      layer.weight *= 1

parameters = [C] + [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

import torch.optim as optim

optimizer = optim.Adam(parameters, lr=1e-4)  # Initial LR
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=150000, gamma=0.1)

82752


In [14]:
@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  x = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  for layer in layers:
    x = layer(x)
  loss = F.cross_entropy(x, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

#we expect around 3, if the initialization is good the probability given to the good token is 1/20 that is e-3

train 3.0462937355041504
val 3.0467724800109863


In [15]:
g = torch.Generator(device=device).manual_seed(1) # for reproducibility
max_steps = 250001
batch_size = 256
lossi = []
ud = []

lri = (torch.logspace(-4,0,max_steps))


for i in range(max_steps):

  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g, device = device)
  Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

  # forward pass
  emb = C[Xb] # embed the characters into vectors
  x = emb.view(emb.shape[0], -1) # concatenate the vectors
  for layer in layers:
    x = layer(x)
  loss = F.cross_entropy(x, Yb) # loss function

  # backward pass
  #optimizer.zero_grad()

  for p in parameters:
    p.grad = None
  loss.backward()


  # update
  #optimizer.step()
  #scheduler.step()

  lr = 0.01 if i < 150000 else 0.001 # step learning rate decay
  for p in parameters:
    p.data += -lr * p.grad

  # track stats
  if i % 10000 == 0: # print every once in a while
    print('epoch    ',f'{i}')
    split_loss('val')
    split_loss('train')
  lossi.append(loss.log10().item())
  with torch.no_grad():
    ud.append([((lr*p.grad).std() / p.data.std()).log10().item() for p in parameters])

  #if i >= 10000:
   # break # AFTER_DEBUG: would take out obviously to run full optimization

epoch     0
val 3.046196222305298
train 3.045710563659668
epoch     10000
val 1.41068434715271
train 1.3192267417907715
epoch     20000
val 1.311635971069336
train 1.1973735094070435
epoch     30000
val 1.2802255153656006
train 1.1469769477844238
epoch     40000
val 1.2578277587890625
train 1.1139968633651733
epoch     50000
val 1.2468727827072144
train 1.088836669921875
epoch     60000
val 1.2389799356460571
train 1.071405053138733
epoch     70000
val 1.233339786529541
train 1.0578527450561523
epoch     80000
val 1.2315073013305664
train 1.047393560409546
epoch     90000
val 1.22542405128479
train 1.0341421365737915
epoch     100000
val 1.2291327714920044
train 1.0313270092010498
epoch     110000
val 1.2255598306655884
train 1.0215574502944946
epoch     120000
val 1.2238937616348267
train 1.0120279788970947
epoch     130000
val 1.2213577032089233
train 1.0069829225540161
epoch     140000
val 1.2227495908737183
train 1.003462553024292
epoch     150000
val 1.22115159034729
train 0.99772

In [17]:
split_loss('train')
split_loss('val')

train 0.9632715582847595
val 1.209210991859436


In [ ]:
lolol

AttributeError: 'list' object has no attribute 'shape'